In [64]:
import pickle
import os
import sys
import plotly.graph_objs as go

path = os.getcwd().split(os.sep +'GUI')[0]
if path not in sys.path:
    sys.path.append(path)

import neurolib.dashboard.layout as layout
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data

import plotly.express as px
from jupyter_dash import JupyterDash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost

%load_ext autoreload
%autoreload 2 

path = os.path.join(os.getcwd(), "plots_")

aln = ALNModel()
data.set_parameters(aln)

state_vars = aln.state_vars
readpath = '.' + os.sep + 'data_final' + os.sep
case = '1'

background_markersize = layout.background_markersize
markersize = layout.markersize

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
### LOAD BISTABLE STATE
with open('bi_granular.pickle','rb') as file:
    load_array= pickle.load(file)
bi_exc = load_array[0]
bi_inh = load_array[1]

##### LOAD BOUNDARIES
with open('boundary_bi_granular.pickle','rb') as file:
    load_array= pickle.load(file)
boundary_bi_exc = load_array[0]
boundary_bi_inh = load_array[1]

with open('boundary_LC_granular.pickle','rb') as file:
    load_array= pickle.load(file)
boundary_LC_exc = load_array[0]
boundary_LC_inh = load_array[1]

with open('boundary_LCbi_granular.pickle','rb') as file:
    load_array= pickle.load(file)
boundary_LC_up_exc = load_array[0]
boundary_LC_up_inh = load_array[1]

In [69]:
tasks = ['Task 1: down to up, sparsity constraints',
         'Task 2: down to up, energy constraints',
         'Task 3: up to down, sparsity constraints',
         'Task 4: up to down, energy constraints']

with open('.' + os.sep + 'bi.pickle','rb') as file:
    load_array= pickle.load(file)
ext_exc = load_array[0]
ext_inh = load_array[1]

data_array = data.read_data(aln, readpath, case)
exc_, inh_, both_c_, no_c_ = data_array[0:4]
exc_1, inh_1, lenx_1, leny_1, exc_2, inh_2, lenx_2, leny_2 = data_array[4:12]
exc_3, inh_3, lenx_3, leny_3, exc_4, inh_4, lenx_4, leny_4, cost1, cost2, cost3, cost4 = data_array[12:24]

data1, data2, data3, data4 = data.get_scatter_data(exc_1, inh_1, exc_2, inh_2, exc_3, inh_3, exc_4, inh_4)
data_background = data.get_data_background(exc_1, inh_1, exc_2, inh_2, exc_3, inh_3, exc_4, inh_4)
trace00, trace01 = data.get_step_current_traces(aln)
trace10, trace11 = layout.get_empty_traces()

bistable_regime = layout.get_bistable_paths(boundary_bi_exc, boundary_bi_inh)
oscillatory_regime = layout.get_osc_path(boundary_LC_exc, boundary_LC_inh)
LC_up_regime = layout.get_LC_up_path(boundary_LC_up_exc, boundary_LC_up_inh)

fig_bifurcation = go.FigureWidget(data=[data_background, data1, data2])#, data3, data4])
fig_bifurcation.update_layout(shapes=[bistable_regime, oscillatory_regime, LC_up_regime])
fig_bifurcation.add_annotation(layout.get_label_bistable())
fig_bifurcation.add_annotation(layout.get_label_osc())
fig_bifurcation.add_annotation(layout.get_label_osc_up())
fig_bifurcation.add_annotation(layout.get_label_down())
fig_bifurcation.add_annotation(layout.get_label_up())
fig_bifurcation.update_layout(layout.get_layout_bifurcation())

fig_time_series_exc = go.Figure(data=[trace00, trace10])
fig_time_series_exc.update_layout(layout.get_layout_exc())
fig_time_series_inh = go.Figure(data=[trace01, trace11])
fig_time_series_inh.update_layout(layout.get_layout_inh())

df = px.data.tips()
app = JupyterDash(__name__)
app.layout = html.Div([
    html.H1("State switching task in bistable regime"),
    html.Div([
        html.H3("Task"),
        html.Label([
              dcc.Dropdown(
                  id='task-dropdown', clearable=False,
                  value=tasks[0], options=[
                      {'label': t, 'value': t}
                      for t in tasks
                  ])
          ]),
        html.H3("Cost"),
        dcc.Graph(id='bifurcation_diagram',
                 figure=fig_bifurcation,
                 ),        
    ], style={'width': '49%', 'display': 'inline-block', 'padding': '0 20'}),
    html.Div([
        html.H3("Time series for step current stimulation"),
        html.Div([
            html.H3("Excitatory node"),
            dcc.Graph(id='time_series_exc',
                 figure=fig_time_series_exc,
                 ),
            ], style={'width': '48%', 'display': 'inline-block', 'padding': '0 20'}),
        html.Div([
            html.H3("Inhibitory node"),
            dcc.Graph(id='time_series_inh',
                 figure=fig_time_series_inh,
                 ),
            ], style={'width': '48%', 'float': 'right', 'display': 'inline-block'}),
    ], style={'width': '49%', 'float': 'right', 'display': 'inline-block'}),
    html.Label([
        "colorscale",
        dcc.Dropdown(
            id='colorscale-dropdown', clearable=False,
            value='plasma', options=[
                {'label': c, 'value': c}
                for c in px.colors.named_colorscales()
            ])
    ]),
    dcc.Graph(id='graph'),
])

@app.callback(
    Output('graph', 'figure'),
    [Input("colorscale-dropdown", "value")]
)
def update_figure(colorscale):
    return px.scatter(
        df, x="total_bill", y="tip", color="size",
        color_continuous_scale=colorscale,
        render_mode="webgl"
    )
    

@app.callback(
        Output('bifurcation_diagram', 'figure'),
        #Output('time_series_exc', 'figure')],
        #Output('time_series_inh', 'figure'),
        [Input('bifurcation_diagram', 'clickData')])
def set_marker(selection):
    
    if selection != None:
        pInd = selection['points'][0]['pointIndex']
        trace = fig_bifurcation.data[selection['points'][0]['curveNumber']]

        functions.setdefaultmarkersize(0, fig_bifurcation.data[0])
        for fig_ in fig_bifurcation.data[1:]:
            functions.setdefaultmarkersize(markersize, fig_)
        functions.setmarkersize(pInd, background_markersize, trace)
    
    return fig_bifurcation

@app.callback(
        Output('time_series_exc', 'figure'),
        #Output('time_series_inh', 'figure'),
        [Input('bifurcation_diagram', 'clickData')])
def plot_trace_exc(selection):
        
    data_ = fig_time_series_exc.data
    layout_ = fig_time_series_exc.layout
    
    if selection != None:        
        time_, exc_trace_, inh_trace_ = data.trace_step(aln, selection['points'][0]['x'], selection['points'][0]['y'])
        
        data_[1].x = time_
        data_[1].y = exc_trace_       
        
    print(fig_time_series_exc.data[1].y)
    fig_time_series_exc.update_layout(layout.get_layout_exc())
    
    return fig_time_series_exc

app.run_server()

case =  .\data_final 1
Dash app running on http://127.0.0.1:8050/
()
[3.69560460e-04 3.52185916e+01 2.61509663e+01 ... 6.14786363e+01
 6.14786363e+01 6.14786363e+01]
